# 4.2 · Servidores MCP externos

**Problema:** el agente necesita leer la base de conocimiento (y en un caso real, Jira, GitHub, la base de datos…). No queremos reescribir esas integraciones.

**Al terminar tendrás:** un agente con un servidor MCP externo (por stdio) junto al servidor in-process.

MCP es un estándar: si alguien ha hecho un servidor MCP para tu herramienta, lo conectas y listo.

## Preparación

Las herramientas del 4.1 ya están en `taller_utils.py`. Necesitamos Node.js: el servidor externo se lanza con `npx` (la primera vez descarga el paquete y tarda un poco).

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import herramientas
from taller_utils import imprimir, buscar_incidencias, crear_incidencia
from claude_agent_sdk import query, create_sdk_mcp_server, ClaudeAgentOptions, SystemMessage

herramientas.restaurar_datos()
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia])

## Qué cambia respecto al in-process

```
in-process                              externo (stdio)
┌───────────── kernel de Python ──┐     ┌──── kernel ────┐        ┌──── otro proceso ────┐
│ agente ──► buscar_incidencias() │     │ agente ────────┼─stdin─►│ servidor de ficheros │
└─────────────────────────────────┘     └────────────────┘◄stdout─┴──────────────────────┘
```

- **In-process:** una función de nuestro programa.
- **Externo:** otro programa, al que se habla por stdio (entrada y salida estándar) o por HTTP.
- Mismo protocolo y mismo nombre `mcp__<servidor>__<herramienta>`: **para el agente no hay diferencia**.

## Conectar un servidor de sistema de ficheros

Usamos el servidor oficial de ficheros de MCP, apuntando a `datos/kb/`. Se describe con el comando que lo lanza:

La ruta va en **absoluto** con `Path(...).resolve()`: el servidor es otro proceso y no conviene que dependa de la carpeta desde la que se ejecuta el notebook.

In [ ]:
KB = Path("../../datos/kb").resolve()

kb = {
    "type": "stdio",
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", str(KB)],
}

### ¿Qué herramientas expone?

Antes de rellenar `allowed_tools` hay que saber qué ofrece el servidor. Lanzamos una consulta trivial y miramos el primer mensaje del stream, el `SystemMessage` de inicio: en `data["tools"]` están todas las herramientas disponibles.

In [ ]:
options_exploracion = ClaudeAgentOptions(
    model=MODEL, mcp_servers={"soporte": soporte, "kb": kb},
    tools=[], max_turns=1, setting_sources=[],
)

async for msg in query(prompt="Responde solo OK", options=options_exploracion):
    if isinstance(msg, SystemMessage) and msg.subtype == "init":
        print(msg.data["mcp_servers"])
        herramientas_disponibles = msg.data["tools"]

[t for t in herramientas_disponibles if t.startswith("mcp__kb__")]

Fíjate: además de leer, el servidor puede **escribir, editar y mover** ficheros (`write_file`, `edit_file`, `move_file`…). Nuestro agente solo necesita leer y buscar, así que solo habilitamos esas:

In [ ]:
options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=("Eres un agente de soporte de nivel 1. Respondes en español y de forma breve. "
                   "Cuando propongas una solución, basa la respuesta en un artículo de la base de "
                   "conocimiento y cítalo."),
    mcp_servers={
        "soporte": soporte,  # in-process
        "kb": kb,            # externo, stdio
    },
    tools=[],
    allowed_tools=[
        "mcp__soporte__buscar_incidencias",
        "mcp__soporte__crear_incidencia",
        "mcp__kb__list_directory",
        "mcp__kb__read_text_file",
        "mcp__kb__search_files",
    ],
    max_turns=10,
    setting_sources=[],
)

In [ ]:
async for msg in query(prompt="Un cliente no puede exportar a PDF, ¿hay algún artículo que lo explique?",
                       options=options):
    imprimir(msg)

El agente busca en la base de conocimiento, lee el artículo y lo cita.

## Criterio

- Un servidor externo puede exponer **muchas** herramientas, incluidas de escritura. Solo se habilitan las necesarias.
- Los nombres de las herramientas dependen del servidor: **compruébalos, no los adivines**.
- Para servidores remotos hay variantes por HTTP, con `url` y `headers`:

```python
"jira": {
    "type": "http",            # o "sse"
    "url": "https://mcp.ejemplo.com/mcp",
    "headers": {"Authorization": f"Bearer {os.environ['JIRA_TOKEN']}"},
}
```

En `taller_utils.py` queda `servidor_kb()`, que devuelve esta misma configuración.

## Resumen

- Un servidor MCP externo es otro proceso (stdio) o un servicio (HTTP). Para el agente, es igual que uno in-process.
- Rutas absolutas para todo lo que se pasa a otro proceso.
- Antes de habilitar herramientas, listarlas (`SystemMessage` de inicio) y elegir solo las necesarias.

**Para el siguiente vídeo:** ya tenemos muchas herramientas. Y cuantas más tiene un agente, más se dispersa.